In [ ]:
# Install Required Libraries
pip install pdfplumber python-docx openpyxl

scrape text from PDF and DOCX files for training an LLM

project/
├── documents/           # Put your PDF/DOCX files here
│   ├── file1.pdf
│   ├── file2.docx
│   └── ...
├── training_data/
│   ├── all_text.txt     # Combined text
│   ├── documents.json  # JSON format
│   └── training/
│       ├── chunks.txt           # Text chunks
│       └── training_pairs.json # Training pairs
└── extractor.py         # This script

In [ ]:
import os
import json
import re
import pdfplumber
from docx import Document
from pathlib import Path
from typing import List, Dict, Optional
from dataclasses import dataclass, asdict
import logging

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

@dataclass
class TextDocument:
    """Data class to store extracted text with metadata"""
    filename: str
    file_type: str
    text: str
    page_count: Optional[int] = None
    word_count: int = 0
    
    def to_dict(self):
        return asdict(self)


class TextExtractor:
    """Main class for extracting text from various file formats"""
    
    def __init__(self):
        self.supported_formats = ['.pdf', '.docx', '.doc']
        self.text_data: List[TextDocument] = []
    
    def extract_from_pdf(self, file_path: str) -> TextDocument:
        """Extract text from PDF file"""
        try:
            text_content = []
            page_count = 0
            
            with pdfplumber.open(file_path) as pdf:
                page_count = len(pdf.pages)
                
                for page_num, page in enumerate(pdf.pages, 1):
                    page_text = page.extract_text()
                    if page_text:
                        text_content.append(f"[Page {page_num}]\n{page_text}")
            
            full_text = "\n\n".join(text_content)
            filename = os.path.basename(file_path)
            
            logger.info(f"Extracted {len(text_content)} pages from {filename}")
            
            return TextDocument(
                filename=filename,
                file_type='pdf',
                text=full_text,
                page_count=page_count,
                word_count=len(full_text.split())
            )
            
        except Exception as e:
            logger.error(f"Error extracting from PDF {file_path}: {e}")
            raise
    
    def extract_from_docx(self, file_path: str) -> TextDocument:
        """Extract text from DOCX file"""
        try:
            doc = Document(file_path)
            text_content = []
            
            # Extract from paragraphs
            for para in doc.paragraphs:
                if para.text.strip():
                    text_content.append(para.text)
            
            # Extract from tables
            for table in doc.tables:
                for row in table.rows:
                    for cell in row.cells:
                        if cell.text.strip():
                            text_content.append(cell.text)
            
            full_text = "\n\n".join(text_content)
            filename = os.path.basename(file_path)
            
            logger.info(f"Extracted text from {filename}")
            
            return TextDocument(
                filename=filename,
                file_type='docx',
                text=full_text,
                page_count=None,
                word_count=len(full_text.split())
            )
            
        except Exception as e:
            logger.error(f"Error extracting from DOCX {file_path}: {e}")
            raise
    
    def extract_from_file(self, file_path: str) -> TextDocument:
        """Extract text based on file extension"""
        ext = Path(file_path).suffix.lower()
        
        if ext == '.pdf':
            return self.extract_from_pdf(file_path)
        elif ext in ['.docx', '.doc']:
            return self.extract_from_docx(file_path)
        else:
            raise ValueError(f"Unsupported file format: {ext}")
    
    def extract_from_directory(self, directory: str) -> List[TextDocument]:
        """Extract text from all supported files in a directory"""
        extracted_docs = []
        
        for root, dirs, files in os.walk(directory):
            for file in files:
                file_path = os.path.join(root, file)
                ext = Path(file).suffix.lower()
                
                if ext in self.supported_formats:
                    try:
                        doc = self.extract_from_file(file_path)
                        extracted_docs.append(doc)
                        self.text_data.append(doc)
                    except Exception as e:
                        logger.warning(f"Skipping {file_path}: {e}")
        
        return extracted_docs


class TextCleaner:
    """Class for cleaning and preprocessing extracted text"""
    
    @staticmethod
    def clean_text(text: str) -> str:
        """Clean and normalize text"""
        # Remove multiple newlines
        text = re.sub(r'\n{3,}', '\n\n', text)
        
        # Remove multiple spaces
        text = re.sub(r' {2,}', ' ', text)
        
        # Remove page numbers and headers (common patterns)
        text = re.sub(r'Page \d+', '', text)
        text = re.sub(r'-\s*\d+\s*-', '', text)
        
        # Remove special characters but keep punctuation
        text = re.sub(r'[^\w\s.,!?;:\'\"-]', '', text)
        
        # Remove leading/trailing whitespace per line
        lines = [line.strip() for line in text.split('\n')]
        text = '\n'.join(lines)
        
        return text.strip()
    
    @staticmethod
    def split_into_chunks(text: str, chunk_size: int = 1000, overlap: int = 100) -> List[str]:
        """Split text into chunks for training"""
        words = text.split()
        chunks = []
        
        for i in range(0, len(words), chunk_size - overlap):
            chunk = ' '.join(words[i:i + chunk_size])
            if chunk:
                chunks.append(chunk)
        
        return chunks
    
    @staticmethod
    def create_training_pairs(text: str, context_window: int = 500) -> List[Dict]:
        """Create training data pairs for chat model"""
        words = text.split()
        pairs = []
        
        for i in range(0, len(words) - context_window, context_window):
            input_text = ' '.join(words[i:i + context_window])
            output_text = ' '.join(words[i + context_window:i + context_window * 2])
            
            if output_text:
                pairs.append({
                    'input': input_text,
                    'output': output_text
                })
        
        return pairs


class TrainingDataExporter:
    """Export extracted text in various formats for training"""
    
    def __init__(self, extractor: TextExtractor, cleaner: TextCleaner):
        self.extractor = extractor
        self.cleaner = cleaner
    
    def export_to_txt(self, output_path: str, cleaned: bool = True):
        """Export all text to a single TXT file"""
        all_text = []
        
        for doc in self.extractor.text_data:
            text = doc.text
            if cleaned:
                text = self.cleaner.clean_text(text)
            all_text.append(f"=== {doc.filename} ===\n{text}\n")
        
        with open(output_path, 'w', encoding='utf-8') as f:
            f.write('\n'.join(all_text))
        
        logger.info(f"Exported to {output_path}")
    
    def export_to_json(self, output_path: str, cleaned: bool = True):
        """Export to JSON format with metadata"""
        data = []
        
        for doc in self.extractor.text_data:
            text = doc.text
            if cleaned:
                text = self.cleaner.clean_text(text)
            
            data.append({
                'filename': doc.filename,
                'file_type': doc.file_type,
                'text': text,
                'word_count': doc.word_count
            })
        
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        
        logger.info(f"Exported to {output_path}")
    
    def export_for_training(self, output_dir: str, chunk_size: int = 1000):
        """Export in training-ready format"""
        os.makedirs(output_dir, exist_ok=True)
        
        all_chunks = []
        all_pairs = []
        
        for doc in self.extractor.text_data:
            cleaned_text = self.cleaner.clean_text(doc.text)
            
            # Create chunks
            chunks = self.cleaner.split_into_chunks(cleaned_text, chunk_size)
            all_chunks.extend(chunks)
            
            # Create training pairs
            pairs = self.cleaner.create_training_pairs(cleaned_text)
            all_pairs.extend(pairs)
        
        # Save chunks
        with open(f"{output_dir}/chunks.txt", 'w', encoding='utf-8') as f:
            f.write('\n---\n'.join(all_chunks))
        
        # Save training pairs as JSON
        with open(f"{output_dir}/training_pairs.json", 'w', encoding='utf-8') as f:
            json.dump(all_pairs, f, ensure_ascii=False, indent=2)
        
        logger.info(f"Exported {len(all_chunks)} chunks and {len(all_pairs)} training pairs")


# Main execution function
def main():
    """Main function to run the extraction"""
    
    # Configuration
    INPUT_DIR = "./documents"           # Directory containing PDF/DOCX files
    OUTPUT_DIR = "./ training_data"    # Output directory
    
    # Create output directory
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    os.makedirs(INPUT_DIR, exist_ok=True)
    
    # Initialize extractor
    extractor = TextExtractor()
    cleaner = TextCleaner()
    
    # Extract from directory
    logger.info(f"Extracting text from {INPUT_DIR}")
    docs = extractor.extract_from_directory(INPUT_DIR)
    
    logger.info(f"Extracted {len(docs)} documents")
    
    # Show summary
    for doc in docs:
        logger.info(f"  - {doc.filename}: {doc.word_count} words")
    
    # Export
    exporter = TrainingDataExporter(extractor, cleaner)
    
    # Export to various formats
    exporter.export_to_txt(f"{OUTPUT_DIR}/all_text.txt", cleaned=True)
    exporter.export_to_json(f"{OUTPUT_DIR}/documents.json", cleaned=True)
    exporter.export_for_training(f"{OUTPUT_DIR}/training", chunk_size=1000)
    
    logger.info("Extraction complete!")
    
    return extractor.text_data


if __name__ == "__main__":
    main()

# Usage Example

In [ ]:
# Quick usage example
from text_extractor import TextExtractor, TextCleaner, TrainingDataExporter

# Initialize
extractor = TextExtractor()

# Extract from single file
pdf_doc = extractor.extract_from_pdf("document.pdf")
docx_doc = extractor.extract_from_docx("document.docx")

# Clean text
cleaner = TextCleaner()
cleaned_text = cleaner.clean_text(pdf_doc.text)

# Create training chunks
chunks = cleaner.split_into_chunks(cleaned_text, chunk_size=500)

# Export
exporter = TrainingDataExporter(extractor, cleaner)
exporter.export_for_training("./training_output")

This script will:

✅ Extract text from PDF and DOCX files

✅ Clean and normalize the text

✅ Split into training chunks

✅ Export in multiple formats ready for LLM training